### Multi-Agent Bidding Negotiation Training Pipeline

**Project:** Enterprise Support Router

This Colab notebook demonstrates the TRL GRPO + Unsloth training pipeline for our 4-Agent Negotiation Protocol. We leverage **Unsloth** for 4-bit Llama-3.2-1B inference, and **TRL** to optimize 4 distinct agent personas collaboratively.

In [ ]:
# 1. Install Requirements
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install openenv-py

### 2. Load the Unsloth 4-bit LLM Backbone

In [ ]:
import torch
from unsloth import FastLanguageModel

print("Loading Unsloth optimized Llama 3.2 1B Instruct model in 4-bit mode...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
model = FastLanguageModel.for_training(model)
print("Model loaded successfully!")

### 3. Initialize the Multi-Agent GRPO Pipeline
Because our OpenEnv requires a strict 3-Phase State Machine, we generate structured synthetic JSON trajectories for the Multi-Agent Personas to optimize against.

In [ ]:
import os
import sys

# Clone the entire project repository to ensure all local imports work
!git clone https://github.com/RavichandraNayakar/openenv-hackathon-project.git
%cd openenv-hackathon-project
sys.path.append(os.getcwd())

print("Cloned repository and set up paths for training.")


### 4. Run the Training Protocol
We train the 4 underlying Agent Personas sequentially using GRPO.

In [ ]:
from my_env.pytorch.training.trl_multi_agent_trainer import MultiAgentGRPOTrainer

# DRY RUN CONFIGURATION
trainer = MultiAgentGRPOTrainer(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    learning_rate=1e-4,
    batch_size=2,          # Keep small to avoid Out Of Memory (OOM) on first test
    num_train_epochs=1,    # Only do 1 epoch for the dry run!
)
print("Initializing GRPO alignment...")

# Kick off the dry run
trainer.train_all_agents(output_dir="./checkpoints_multi_agent")
